# House Price Prediction (King County / Seattle area, 2014)
End-to-end **regression** workflow: data loading → exploration → EDA → preprocessing → feature engineering → data visualization → model training → model evaluation.

**Target:** `price` (sale price in USD). Place `data.csv` next to this notebook or edit `DATA_PATH`.

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.ticker import FuncFormatter

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error, mean_absolute_percentage_error

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)
RANDOM_STATE = 42
DATA_PATH = "data.csv"   # change if the file is elsewhere
usd = FuncFormatter(lambda x, _: f"${x/1e6:.1f}M" if x >= 1e6 else f"${x/1e3:.0f}k")

## 1. Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

## 2. Data Exploration

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.describe(include=['object', 'string']).T

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", df.duplicated().sum())
print("Sale dates:", df.date.min(), "→", df.date.max())
print("Unique cities:", df.city.nunique(), "| unique zip codes:", df.statezip.nunique())
print("sqft_living == sqft_above + sqft_basement for all rows:",
      bool((df.sqft_living == df.sqft_above + df.sqft_basement).all()))

There are no nulls, but the summary statistics hint at **hidden problems**: `price` has a minimum of 0, a maximum of $26.6M, and bedrooms/bathrooms can be 0. Let's inspect them.

In [ ]:
df["price_per_sqft"] = df.price / df.sqft_living
print("price == 0:           ", (df.price == 0).sum())
print("bedrooms == 0:        ", (df.bedrooms == 0).sum())
print("bathrooms == 0:       ", (df.bathrooms == 0).sum())
print("price/sqft < $40:     ", ((df.price > 0) & (df.price_per_sqft < 40)).sum())
print("price/sqft > $1,500:  ", (df.price_per_sqft > 1500).sum())
print()
print("Most extreme price-per-sqft rows:")
df.loc[(df.price == 0) | (df.price_per_sqft < 40) | (df.price_per_sqft > 1500),
       ["price", "sqft_living", "bedrooms", "bathrooms", "city", "price_per_sqft"]].sort_values("price_per_sqft").iloc[list(range(0, 3)) + list(range(-3, 0))]

- 49 homes have a **price of $0** — sale records with a missing price, not real sales.
- One 1,180 sqft house in Kent sold for $26.6M (≈ $22,500/sqft) and one 780 sqft home for $7,800 are almost certainly data-entry errors.
- Two rows have 0 bedrooms *and* 0 bathrooms.
- `yr_renovated` is 0 for homes never renovated, so it is a code, not a year.

## 3. Exploratory Data Analysis (EDA)

In [ ]:
eda = df[(df.price > 0) & df.price_per_sqft.between(40, 1500) & (df.bedrooms > 0) & (df.bathrooms > 0)].copy()
print(f"Kept {len(eda)} of {len(df)} rows ({len(df) - len(eda)} removed as invalid)")
print()
print("Price summary ($):")
print(eda.price.describe().round(0).to_string())
print(f"\nSkewness of price: {eda.price.skew():.2f}   |   skewness of log(price): {np.log(eda.price).skew():.2f}")

In [ ]:
num = eda.select_dtypes("number").drop(columns="price_per_sqft")
print("Correlation with price (Pearson):")
num.corr()["price"].drop("price").sort_values(ascending=False).round(3)

In [ ]:
g = eda.groupby("city").agg(n=("price", "size"), median_price=("price", "median"), median_ppsf=("price_per_sqft", "median"))
print("Most expensive cities (≥30 sales), median price:")
g[g.n >= 30].sort_values("median_price", ascending=False).head(8).round(0)

In [ ]:
print("Median price by number of bedrooms / waterfront / renovated:")
display(eda.groupby("bedrooms").price.agg(["median", "count"]).round(0))
display(eda.groupby("waterfront").price.agg(["median", "count"]).round(0))
display(eda.assign(renovated=eda.yr_renovated > 0).groupby("renovated").price.agg(["median", "count"]).round(0))

**EDA findings**
- Price is strongly **right-skewed**; its log is nearly symmetric, so we will model `log(price)`.
- Living area (`sqft_living`) is the strongest numeric predictor, followed by bathrooms, `sqft_above`, bedrooms and view.
- **Location matters hugely**: median prices differ several-fold between cities such as Mercer Island/Bellevue versus the southern suburbs.
- Waterfront homes cost roughly 2.3× more at the median ($1.07M vs $463k), but there are only 30 of them.

## 4. Data Preprocessing

1. Remove invalid records (price 0, implausible price-per-sqft, 0 bed/bath)
2. Drop columns with no information: `country` (single value) and `street` (almost unique per row)
3. Parse the sale date
4. Train/test split (80/20)

Target is modelled as **log(price)**, then converted back to dollars for evaluation.

In [ ]:
clean = df[(df.price > 0) & df.price_per_sqft.between(40, 1500) & (df.bedrooms > 0) & (df.bathrooms > 0)].copy()
clean = clean.drop(columns=["price_per_sqft", "country", "street"]).reset_index(drop=True)
clean["date"] = pd.to_datetime(clean.date)
print("Clean shape:", clean.shape)
clean.dtypes.to_frame("dtype").T

In [ ]:
y = np.log(clean.price)               # modelling target
X_raw = clean.drop(columns="price")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train_raw.shape, "| Test:", X_test_raw.shape)
price_test = np.exp(y_test)           # true dollar prices for the test set

## 5. Feature Engineering

Row-wise features (no information shared across rows, so no leakage), plus **target encoding** of the high-cardinality location columns (`city`, `statezip`). Target encoding is fitted *inside* the model pipeline with internal cross-fitting, so each house's own price never leaks into its own encoding.

In [ ]:
SALE_YEAR = 2014

def engineer(X_in):
    X = X_in.copy()
    X["house_age"]        = SALE_YEAR - X.yr_built
    X["renovated"]        = (X.yr_renovated > 0).astype(int)
    X["years_since_reno"] = np.where(X.yr_renovated > 0, SALE_YEAR - X.yr_renovated, X.house_age)
    X["has_basement"]     = (X.sqft_basement > 0).astype(int)
    X["total_rooms"]      = X.bedrooms + X.bathrooms
    X["sqft_per_room"]    = X.sqft_living / X.total_rooms
    X["bath_per_bed"]     = X.bathrooms / X.bedrooms
    X["lot_to_living"]    = X.sqft_lot / X.sqft_living
    X["log_sqft_living"]  = np.log(X.sqft_living)
    X["log_sqft_lot"]     = np.log(X.sqft_lot)
    X["basement_share"]   = X.sqft_basement / X.sqft_living
    X["quality_score"]    = X.condition * (1 + X.view) * (1 + X.waterfront)
    X["is_new"]           = (X.house_age <= 5).astype(int)
    X["is_historic"]      = (X.house_age >= 80).astype(int)
    X["sale_month"]       = X.date.dt.month
    X["zip"]              = X.statezip.str.replace("WA ", "", regex=False)
    return X.drop(columns=["date", "statezip", "yr_renovated"])

X_train = engineer(X_train_raw)
X_test  = engineer(X_test_raw)

cat_cols = ["city", "zip"]
num_cols = [c for c in X_train.columns if c not in cat_cols]
print("Features:", X_train.shape[1], "(", len(num_cols), "numeric +", len(cat_cols), "target-encoded )")
print(num_cols)

In [ ]:
new = ["house_age", "renovated", "years_since_reno", "has_basement", "total_rooms", "sqft_per_room", "bath_per_bed",
       "lot_to_living", "log_sqft_living", "log_sqft_lot", "basement_share", "quality_score", "is_new", "is_historic"]
print("Correlation of new features with log(price) in the training set:")
pd.concat([X_train[new], y_train.rename("log_price")], axis=1).corr()["log_price"].drop("log_price").sort_values().round(3)

In [ ]:
# Preprocessors: target-encode location; scale numerics only for the linear model
def make_prep(scale):
    num_step = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([
        ("loc", TargetEncoder(target_type="continuous", smooth="auto", random_state=RANDOM_STATE), cat_cols),
        ("num", num_step, num_cols)])

## 6. Data Visualization

Plots use the cleaned dataset (invalid records removed).

### 6.1 Target distribution and outliers

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
sns.histplot(clean.price, bins=60, kde=True, color="#4C72B0", ax=ax[0])
ax[0].xaxis.set_major_formatter(usd); ax[0].set_title(f"Price (skew = {clean.price.skew():.2f})")
sns.histplot(np.log(clean.price), bins=60, kde=True, color="#55A868", ax=ax[1])
ax[1].set_title(f"log(Price) (skew = {np.log(clean.price).skew():.2f})")
removed = df[~df.index.isin(df[(df.price > 0) & df.price_per_sqft.between(40, 1500) & (df.bedrooms > 0) & (df.bathrooms > 0)].index)]
ax[2].scatter(df.sqft_living, df.price.clip(upper=3e6), s=8, alpha=.35, c="#999999", label="kept")
ax[2].scatter(removed.sqft_living, removed.price.clip(upper=3e6), s=30, c="#C44E52", label=f"removed ({len(removed)})")
ax[2].yaxis.set_major_formatter(usd); ax[2].set_xlabel("sqft_living"); ax[2].set_title("Invalid records removed (price clipped at $3M)"); ax[2].legend()
plt.tight_layout(); plt.show()

### 6.2 Feature distributions

In [ ]:
cols = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "condition", "yr_built", "sqft_basement"]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for a, c in zip(axes.ravel(), cols):
    sns.histplot(clean[c], bins=30, color="#4C72B0", ax=a); a.set_title(c)
plt.suptitle("Distributions of the main features", y=1.01, fontsize=15)
plt.tight_layout(); plt.show()

### 6.3 What drives price?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6))
sns.scatterplot(data=clean, x="sqft_living", y="price", hue="waterfront", palette={0: "#4C72B0", 1: "#DD8452"},
                alpha=.5, s=22, ax=ax[0])
ax[0].yaxis.set_major_formatter(usd); ax[0].set_title("Price vs living area")
sns.regplot(x=np.log(clean.sqft_living), y=np.log(clean.price), scatter_kws=dict(alpha=.2, s=10),
            line_kws=dict(color="red"), order=2, ci=None, ax=ax[1])
ax[1].set(xlabel="log(sqft_living)", ylabel="log(price)"); ax[1].set_title("Log–log relationship (quadratic fit)")
sns.scatterplot(data=clean, x="yr_built", y="price", alpha=.35, s=14, color="#4C72B0", ax=ax[2])
ax[2].yaxis.set_major_formatter(usd); ax[2].set_ylim(0, 3e6); ax[2].set_title("Price vs year built")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for a, c in zip(axes.ravel(), ["bedrooms", "bathrooms", "floors", "view", "condition", "waterfront"]):
    d = clean[clean.bedrooms <= 7] if c == "bedrooms" else clean
    d = d[d.bathrooms.isin([1, 1.5, 2, 2.5, 3, 3.5, 4])] if c == "bathrooms" else d
    sns.boxplot(data=d, x=c, y="price", color="#9DB4D8", fliersize=2, ax=a)
    a.set_yscale("log"); a.set_title(f"Price by {c}")
plt.suptitle("Price (log scale) by categorical-like features", y=1.01, fontsize=15)
plt.tight_layout(); plt.show()

### 6.4 Location

In [ ]:
big = clean.city.value_counts()[lambda s: s >= 40].index
c = clean[clean.city.isin(big)]
order = c.groupby("city").price.median().sort_values(ascending=False).index

fig, ax = plt.subplots(1, 2, figsize=(20, 8))
sns.boxplot(data=c, y="city", x="price", order=order, color="#9DB4D8", fliersize=1.5, ax=ax[0])
ax[0].set_xscale("log"); ax[0].set_title("Price by city (cities with ≥40 sales, log scale)")

ppsf = (c.assign(ppsf=c.price / c.sqft_living).groupby("city").ppsf.median().sort_values())
ax[1].barh(ppsf.index, ppsf.values, color="#DD8452")
for i, v in enumerate(ppsf.values): ax[1].text(v + 3, i, f"${v:,.0f}", va="center", fontsize=9)
ax[1].set_title("Median price per sqft by city"); ax[1].set_xlim(0, ppsf.max() * 1.15)
plt.tight_layout(); plt.show()

In [ ]:
z = clean.groupby("statezip").price.agg(["median", "count"])
z = z[z["count"] >= 20].sort_values("median")
plt.figure(figsize=(9, 11))
plt.barh(z.index.str.replace("WA ", ""), z["median"], color=plt.cm.viridis((z["median"] - z["median"].min()) / (z["median"].max() - z["median"].min())))
plt.gca().xaxis.set_major_formatter(usd); plt.xlabel("Median price"); plt.title("Median price by zip code (≥20 sales)")
plt.tight_layout(); plt.show()

### 6.5 Correlations

In [ ]:
cm = clean.drop(columns=["date", "street", "city", "statezip"], errors="ignore").select_dtypes("number").copy()
cm["log_price"] = np.log(cm.price); cm = cm.drop(columns="price")
corr = cm.corr()
fig, ax = plt.subplots(1, 2, figsize=(19, 7))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": .7}, annot_kws={"size": 8}, ax=ax[0])
ax[0].set_title("Correlation matrix")
s = corr["log_price"].drop("log_price").sort_values()
ax[1].barh(s.index, s.values, color=["#C44E52" if v < 0 else "#4C72B0" for v in s.values])
ax[1].axvline(0, color="k", lw=.8); ax[1].set_title("Correlation with log(price)")
plt.tight_layout(); plt.show()

In [ ]:
sns.pairplot(clean.assign(log_price=np.log(clean.price))[["log_price", "sqft_living", "bathrooms", "bedrooms", "yr_built"]],
             corner=True, plot_kws=dict(alpha=.3, s=12), height=2.2)
plt.suptitle("Pairwise relationships of key features", y=1.02, fontsize=14); plt.show()

## 7. Model Training

We compare a regularised linear model against tree ensembles, all predicting `log(price)`.
Cross-validation (5-fold) on the training set uses RMSE on the log scale (lower is better) and R².

In [ ]:
cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Ridge":             Pipeline([("prep", make_prep(True)),  ("m", Ridge(alpha=1.0))]),
    "Random Forest":     Pipeline([("prep", make_prep(False)), ("m", RandomForestRegressor(200, random_state=RANDOM_STATE, n_jobs=1))]),
    "Gradient Boosting": Pipeline([("prep", make_prep(False)), ("m", GradientBoostingRegressor(random_state=RANDOM_STATE))]),
    "HistGradientBoosting": Pipeline([("prep", make_prep(False)), ("m", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
}
cv_rmse, cv_r2 = {}, {}
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"}, n_jobs=-1)
    cv_rmse[name], cv_r2[name] = -r["test_rmse"], r["test_r2"]
    print(f"{name:22s} CV RMSE(log) = {cv_rmse[name].mean():.4f} ± {cv_rmse[name].std():.4f}   CV R² = {cv_r2[name].mean():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=pd.DataFrame(cv_r2), orient="h", palette="Set2", ax=ax[0]); ax[0].set_title("CV R²")
sns.boxplot(data=pd.DataFrame(cv_rmse), orient="h", palette="Set2", ax=ax[1]); ax[1].set_title("CV RMSE on log(price) (lower = better)")
plt.tight_layout(); plt.show()

### Hyperparameter tuning

In [ ]:
ridge_gs = GridSearchCV(models["Ridge"], {"m__alpha": [0.1, 1, 10, 30, 100]},
                        cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)

rf_gs = GridSearchCV(models["Random Forest"],
    {"m__min_samples_leaf": [1, 3], "m__max_features": [0.5, 1.0]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)

gb_gs = GridSearchCV(models["Gradient Boosting"],
    {"m__n_estimators": [300], "m__learning_rate": [0.05], "m__max_depth": [3, 4], "m__subsample": [0.8]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)

hgb_gs = GridSearchCV(models["HistGradientBoosting"],
    {"m__learning_rate": [0.03, 0.06], "m__max_depth": [3, None], "m__max_iter": [300], "m__l2_regularization": [1.0]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)

for n, g in [("Ridge", ridge_gs), ("RF", rf_gs), ("GB", gb_gs), ("HGB", hgb_gs)]:
    print(f"{n:6s} best CV RMSE(log) = {-g.best_score_:.4f}  params = { {k.split('__')[1]: v for k, v in g.best_params_.items()} }")

In [ ]:
final = {"Ridge (tuned)": ridge_gs.best_estimator_, "Random Forest (tuned)": rf_gs.best_estimator_,
         "Gradient Boosting (tuned)": gb_gs.best_estimator_, "HistGradientBoosting (tuned)": hgb_gs.best_estimator_}

## 8. Model Evaluation (held-out test set)

Metrics are computed on **real dollar prices** (predictions converted back with `exp`). MAPE and median error are shown because RMSE in dollars is dominated by a handful of very expensive homes.

In [ ]:
def metrics(y_true_log, pred_log):
    t, p = np.exp(y_true_log), np.exp(pred_log)
    return dict(R2_log=r2_score(y_true_log, pred_log), R2_dollars=r2_score(t, p),
                RMSE=np.sqrt(mean_squared_error(t, p)), MAE=mean_absolute_error(t, p),
                MedianAE=median_absolute_error(t, p), MAPE_pct=mean_absolute_percentage_error(t, p) * 100)

preds = {n: m.predict(X_test) for n, m in final.items()}
res = pd.DataFrame({n: metrics(y_test, p) for n, p in preds.items()}).T.round(3).sort_values("MAE")
res

In [ ]:
best_name = res.index[0]; best_pred = preds[best_name]
print("Best model (lowest MAE):", best_name)
print(f"Typical error: median ${median_absolute_error(price_test, np.exp(best_pred)):,.0f}, "
      f"mean ${mean_absolute_error(price_test, np.exp(best_pred)):,.0f}")
within = np.abs(np.exp(best_pred) / price_test - 1)
for t in (0.10, 0.20, 0.30):
    print(f"Predictions within ±{int(t*100)}% of true price: {(within <= t).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6))
t, p = price_test.values, np.exp(best_pred)
ax[0].scatter(t, p, s=12, alpha=.4, c="#4C72B0"); lim = [1e5, 3e6]
ax[0].plot(lim, lim, "r--"); ax[0].set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="Actual price", ylabel="Predicted price")
ax[0].set_title(f"Actual vs predicted: {best_name}")

resid = best_pred - y_test.values
ax[1].scatter(best_pred, resid, s=12, alpha=.4, c="#55A868"); ax[1].axhline(0, color="r", ls="--")
ax[1].set(xlabel="Predicted log(price)", ylabel="Residual (log scale)", title="Residuals vs predicted")

sns.histplot(resid, bins=40, kde=True, color="#8172B3", ax=ax[2])
ax[2].set_title(f"Residual distribution (mean {resid.mean():.3f}, std {resid.std():.3f})")
plt.tight_layout(); plt.show()

In [ ]:
# Error by price bracket
bins = pd.qcut(price_test, 5, labels=["Lowest 20%", "20-40%", "40-60%", "60-80%", "Top 20%"])
err = pd.DataFrame({"bracket": bins, "abs_pct_err": np.abs(np.exp(best_pred) / price_test - 1) * 100,
                    "signed_pct_err": (np.exp(best_pred) / price_test - 1) * 100})
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=err, x="bracket", y="abs_pct_err", estimator=np.median, color="#DD8452", errorbar=None, ax=ax[0])
ax[0].set_ylabel("Median absolute % error"); ax[0].set_title("Prediction error by price bracket")
sns.boxplot(data=err, x="bracket", y="signed_pct_err", color="#9DB4D8", fliersize=2, ax=ax[1])
ax[1].axhline(0, color="r", ls="--"); ax[1].set_ylim(-80, 120); ax[1].set_ylabel("Signed % error (pred vs actual)")
ax[1].set_title("Bias by price bracket (+ = overpredict)")
plt.tight_layout(); plt.show()

In [ ]:
# Model comparison on test metrics
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
res["R2_dollars"].sort_values().plot.barh(ax=ax[0], color="#4C72B0"); ax[0].set_title("R² (dollars) — higher is better")
res["MAE"].sort_values(ascending=False).plot.barh(ax=ax[1], color="#DD8452"); ax[1].set_title("MAE ($) — lower is better")
res["MAPE_pct"].sort_values(ascending=False).plot.barh(ax=ax[2], color="#55A868"); ax[2].set_title("MAPE (%) — lower is better")
plt.tight_layout(); plt.show()

In [ ]:
# Which features matter? Permutation importance on the test set (model-agnostic, in the best model's own terms)
pi = permutation_importance(final[best_name], X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
                            scoring="neg_root_mean_squared_error", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False).head(15)
sd  = pd.Series(pi.importances_std, index=X_test.columns)[imp.index]
plt.figure(figsize=(9, 6.5))
plt.barh(imp.index[::-1], imp.values[::-1], xerr=sd.values[::-1], color="steelblue")
plt.xlabel("Increase in RMSE (log scale) when feature is shuffled")
plt.title(f"Top 15 permutation importances: {best_name}"); plt.tight_layout(); plt.show()

## Conclusions
- **Data quality first:** 57 invalid records (49 zero-price sales, 6 absurd price-per-sqft values, 2 homes with 0 bedrooms/bathrooms) were removed, and modelling `log(price)` tamed the heavy right skew (4.0 → 0.3).
- **Best model: tuned Gradient Boosting** — test R² = 0.84 on log(price) and 0.80 in dollars, median error ≈ $50k, mean error ≈ $88k, MAPE ≈ 15.5%. About **47% of predictions land within ±10%** of the true price, 76% within ±20% and 88% within ±30%.
- **Tree ensembles beat the linear model**, mainly on expensive homes: Ridge reaches a similar R² on the log scale (0.82) but only 0.54 in dollars (RMSE ≈ $247k vs ≈ $163k), because it cannot capture the non-linear price escalation of large, high-end homes. The three tree models are close to one another on this small test set (909 homes).
- **Location and size dominate.** Permutation importance ranks the target-encoded **zip code** first by a wide margin, then `sqft_living`, city, and the engineered `quality_score`; `sqft_above`, lot size and bath-per-bed follow. Age and renovation features add little.
- **Errors are largest at the extremes:** median absolute error is ≈ 12–13% for the cheapest and the most expensive 20% of homes versus ≈ 9–10% in the middle, and the model slightly over-predicts cheap homes and under-predicts luxury ones.
- **Limitations:** only ~4,500 sales over ~10 weeks, no latitude/longitude, and no build-grade or interior-quality field, so neighbourhood and finish quality are the main sources of the remaining error. Hyperparameter grids were kept small, so some further gain from tuning is likely.
- **Next steps:** add geocoded coordinates, stack the best models, and widen the search grids.